# Задание 4

In [ ]:
import os
import warnings

import boto3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from botocore.client import Config
from sqlalchemy import create_engine, text

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

PG_HOST = os.getenv("POSTGRES_HOST", "localhost")
PG_URL = (
    f"postgresql+psycopg2://{os.getenv('POSTGRES_USER', 'oiluser')}:"
    f"{os.getenv('POSTGRES_PASSWORD', 'oilpass')}@"
    f"{PG_HOST}:{os.getenv('POSTGRES_PORT', '5432')}/"
    f"{os.getenv('POSTGRES_DB', 'oildb')}"
)
engine = create_engine(PG_URL)

def minio_endpoint_url():
    raw = os.getenv("MINIO_ENDPOINT", "localhost:9000").strip().rstrip("/")
    if raw.startswith("http://") or raw.startswith("https://"):
        return raw
    return f"http://{raw}"

def minio_client():
    return boto3.client(
        "s3",
        endpoint_url=minio_endpoint_url(),
        aws_access_key_id=os.getenv("MINIO_ACCESS_KEY", "minioadmin"),
        aws_secret_access_key=os.getenv("MINIO_SECRET_KEY", "minioadmin"),
        config=Config(signature_version="s3v4"),
        region_name="us-east-1",
    )


In [ ]:

d = pd.read_sql("""
    SELECT d.*, dr.name AS driver_name, dr.experience_years,
           ROUND((d.cost_usd / NULLIF(d.distance_km,0))::numeric, 2) AS cost_per_km
    FROM deliveries d
    LEFT JOIN drivers dr ON dr.driver_id = d.driver_id
""", engine)

delay_by_weather = d.groupby("weather_conditions")["delay_hours"].mean().sort_values(ascending=False)
delay_by_driver = d.groupby("driver_name")["delay_hours"].mean().sort_values(ascending=False)
print("Задержка vs погода:\n", delay_by_weather)
print("\nЗадержка vs водитель:\n", delay_by_driver)

corr_dist = d[["distance_km", "delay_hours"]].corr().iloc[0,1]
print(f"\nКорреляция расстояние-задержка: {corr_dist:.3f}")
print("\nCost/km:\n", d["cost_per_km"].describe())

best_routes = d[d.delay_hours==0].nsmallest(5, "cost_per_km")[["source","destination","cost_per_km","distance_km"]]
print("\nЛучшие маршруты (min cost/km, без задержки):\n", best_routes)



In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.boxplot(data=d, x="weather_conditions", y="delay_hours", ax=axes[0])
axes[0].set_title("Delay vs Weather")
axes[0].tick_params(axis="x", rotation=30)

sns.scatterplot(data=d, x="distance_km", y="cost_usd", hue="delay_hours", ax=axes[1])
axes[1].set_title("Cost vs Distance")

driver_kpi = d.groupby("driver_name").agg(
    deliveries=("delivery_id", "count"),
    avg_delay=("delay_hours", "mean"),
    avg_cost_km=("cost_per_km", "mean"),
).reset_index()
driver_kpi.plot.bar(x="driver_name", y="avg_delay", ax=axes[2], legend=False)
axes[2].set_title("KPI по водителям (avg delay)")
axes[2].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

